In [1]:
import pandas as pd
import os
        

In [2]:
caminho = ('../data/raw/dataset_01_parceiroA.csv')

df_teste = pd.read_csv(caminho)
df_teste.head()

,Data,Grupos de usuários,Parceiro,compradores,comissão,cashback,vendas totais
0,2011-01-01,Grupo 1,Parceiro A,196,R$ 10.273,R$ 3.267,R$ 93.390
1,2011-01-02,Grupo 1,Parceiro A,115,R$ 7.555,R$ 2.060,R$ 68.682
2,2011-01-03,Grupo 1,Parceiro A,82,R$ 4.839,R$ 1.358,R$ 43.993
3,2011-01-04,Grupo 1,Parceiro A,172,R$ 10.419,R$ 2.907,R$ 94.722
4,2011-01-05,Grupo 1,Parceiro A,187,R$ 11.305,R$ 3.138,R$ 102.768


In [ ]:
def carregar_data_clean(filepath):
 
    df = pd.read_csv(filepath)
    # df['Data'] = pd.to_datetime(df['Data'], format='%d-%m-%Y', errors='coerce')
    
    
    cols_financeiras = ['comissão', 'cashback', 'vendas totais']
    
    for col in cols_financeiras:
        if col in df.columns:
            # Removendo o R$, os espaços em branco, tira os pontos de milhar e troca a vírgula decimal
            df[col] = (df[col]
                       .astype(str)
                       .str.replace('R$', '', regex=False)
                       .str.replace('.', '', regex=False)
                       .str.replace(',', '.', regex=False)
                       .str.strip())
            # Converte para float
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            
    if 'compradores' in df.columns:
        df['compradores'] = df['compradores'].fillna(0).astype(int)
        
    return df

def calcular_metricas(df):
  
    # Agrupa os dados por Grupo 1, 2, 3 e soma os resultados do período
    df_agrupamento = df.groupby('Grupos de usuários').agg({
        'compradores': 'sum',
        'comissão': 'sum',
        'cashback': 'sum',
        'vendas totais': 'sum'
    }).reset_index()
    
    
    df_agrupamento['receita_liquida'] = df_agrupamento['comissão'] - df_agrupamento['cashback']
    
   #Qual foi o retorno percentual sobre o cashback investido?
    df_agrupamento['ret_percentual'] = df_agrupamento.apply(
        lambda row: (row['receita_liquida'] / row['cashback'] * 100) if row['cashback'] > 0 else 0, 
        axis=1
    )
    
    # Arredondamento
    df_agrupamento = df_agrupamento.round(2)
    
    return df_agrupamento

def processar_teste_ab(filepath):
    
    df_clean = carregar_data_clean(filepath)
    df_sumario = calcular_metricas(df_clean)
    return df_sumario

# Bloco de teste 
if __name__ == "__main__":
    
    if os.path.exists(caminho):
        resultado = processar_teste_ab(caminho)
        print("Resumo Consolidado do Teste A/B:")
        print(resultado.to_markdown(index=False)) # Transformar em MD para leitura da IA
    else:
        print(f"Arquivo não encontrado: {caminho}.")

    

Resumo Consolidado do Teste A/B:
| Grupos de usuários   |   compradores |   comissão |   cashback |   vendas totais |   receita_liquida |   ret_percentual |
|:---------------------|--------------:|-----------:|-----------:|----------------:|------------------:|-----------------:|
| Grupo 1              |          9633 |     638135 |     233424 |         5605173 |            404711 |           173.38 |
| Grupo 2              |         10814 |     728178 |     370659 |         6423096 |            357519 |            96.45 |
| Grupo 3              |         11410 |     767887 |     503600 |         6785856 |            264287 |            52.48 |
